# EWS Fraud Detection - Tahap 4: Deteksi Anomali Menggunakan Isolation Forest

Notebook ini mengimplementasikan deteksi anomali tanpa pengawasan (unsupervised anomaly detection) menggunakan model **Isolation Forest** pada data keuangan hasil Tahap 3B (`Fraud_Dataset_Final.xlsx`).

### Alur Pengerjaan:
1. **Penyaringan Data & Imputasi**: 
   - Menghapus observasi yang tidak memiliki metrik utama (misalnya baris tahun dasar 2021).
   - Melakukan imputasi nilai kosong (`NaN`) menggunakan **Median Imputer** (`SimpleImputer`).
2. **Standardisasi Robust**: 
   - Mentransformasi fitur menggunakan **RobustScaler** (menggunakan median dan IQR, sehingga tidak sensitif terhadap pencilan data keuangan).
3. **Isolation Forest Multi-Skenario**:
   - Menjalankan model untuk 3 tingkat kontaminasi (proposi anomali terduga): **3% (0.03)**, **5% (0.05)**, dan **10% (0.10)**.
   - Menyimpan `anomaly_label` (1 = normal, -1 = anomali) dan `anomaly_score` (skor tingkat keparahan anomali) untuk masing-masing skenario.

### Fitur yang Digunakan:
- `dsri`, `gmi`, `aqi`, `sgi`, `lvgi`, `tata` (Komponen Beneish M-Score)
- `revenue_growth`, `asset_growth`, `net_income_growth_assets` (Fitur Tren Keuangan)
- `cfo_to_net_income` (Rasio Arus Kas)

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest

in_path = "Fraud_Dataset_Final.xlsx"
out_path = "Fraud_Dataset_Anomaly.xlsx"

## 1. Memuat Data & Penyaringan Data Panel
Menyaring baris data yang kosong pada metrik kritis (seperti tahun 2021 yang tidak memiliki data pembanding) sebelum melakukan imputasi.

In [3]:
if not os.path.exists(in_path):
    print(f"ERROR: File {in_path} tidak ditemukan!")
else:
    df = pd.read_excel(in_path)
    
    # Saring baris yang tidak memiliki komponen kritis
    critical_features = ['dsri', 'gmi', 'aqi', 'sgi']
    df_clean = df.dropna(subset=critical_features).copy()
    
    print(f"Bentuk Data Awal: {df.shape}")
    print(f"Bentuk Data Setelah Penyaringan (Tahun 2022-2024): {df_clean.shape}")

Bentuk Data Awal: (2466, 57)
Bentuk Data Setelah Penyaringan (Tahun 2022-2024): (1633, 57)


## 2. Imputasi & Standardisasi Robust
Mengisi nilai kosong tersisa menggunakan Median Imputer dan menerapkan standardisasi RobustScaler.

In [4]:
features = [
    'dsri',
    'gmi',
    'aqi',
    'sgi',
    'lvgi',
    'tata',
    'revenue_growth',
    'asset_growth',
    'net_income_growth_assets',
    'cfo_to_net_income'
]

# 1. Median Imputation
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(df_clean[features])
print("Imputasi Nilai Kosong Selesai.")

# 2. Robust Standardization
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
print("Standardisasi Robust Selesai.")

Imputasi Nilai Kosong Selesai.
Standardisasi Robust Selesai.


## 3. Eksekusi Isolation Forest (Tingkat Kontaminasi: 3%, 5%, 10%)
Melatih model Isolation Forest untuk mengidentifikasi pencilan pencatatan transaksi yang mencurigakan.

In [5]:
contaminations = [0.03, 0.05, 0.10]
print("=== MENJALANKAN ISOLATION FOREST ===")

for cont in contaminations:
    suffix = f"{int(cont*100):02d}"
    iso = IsolationForest(
        n_estimators=300,
        contamination=cont,
        random_state=42
    )
    
    # Fit model dan prediksi label
    df_clean[f'anomaly_label_{suffix}'] = iso.fit_predict(X_scaled)
    # Skor anomali (semakin positif mendekati 1.0 semakin anomali)
    df_clean[f'anomaly_score_{suffix}'] = -iso.score_samples(X_scaled)
    
    # Analisis jumlah anomali
    anomaly_count = (df_clean[f'anomaly_label_{suffix}'] == -1).sum()
    pct = (anomaly_count / len(df_clean)) * 100
    print(f"Kontaminasi {cont:.2f}: Terdeteksi {anomaly_count} anomali ({pct:.2f}% dari data)")

=== MENJALANKAN ISOLATION FOREST ===
Kontaminasi 0.03: Terdeteksi 49 anomali (3.00% dari data)
Kontaminasi 0.05: Terdeteksi 82 anomali (5.02% dari data)
Kontaminasi 0.10: Terdeteksi 164 anomali (10.04% dari data)


## 4. Analisis & Ekspor Data Akhir
Menyimpan dataset yang dilengkapi label deteksi anomali untuk analisis fraud di EWS.

In [6]:
# Tampilkan contoh baris yang terdeteksi anomali pada kontaminasi 5%
print("=== Contoh Perusahaan Terdeteksi Anomali (Contamination 5%) ===")
cols_show = ['kode', 'tahun', 'm_score', 'anomaly_score_05', 'anomaly_label_05']
print(df_clean[df_clean['anomaly_label_05'] == -1][cols_show].head(10).to_string(index=False))

# Ekspor
df_clean.to_excel(out_path, index=False)
print(f"\nDataset deteksi anomali sukses disimpan ke '{out_path}'. Bentuk tabel: {df_clean.shape}")

=== Contoh Perusahaan Terdeteksi Anomali (Contamination 5%) ===
kode  tahun   m_score  anomaly_score_05  anomaly_label_05
ALKA   2023  0.104713          0.538349                -1
ALKA   2024  3.383704          0.565601                -1
ARGO   2022 13.873175          0.733239                -1
AYLS   2022  0.486295          0.509027                -1
BAUT   2022  0.575978          0.600248                -1
BOBA   2022  2.420187          0.571194                -1
BSML   2022  2.984983          0.550771                -1
CAKK   2023  2.783652          0.505859                -1
CENT   2022 -3.106150          0.515826                -1
CLAY   2022  1.992274          0.547146                -1

Dataset deteksi anomali sukses disimpan ke 'Fraud_Dataset_Anomaly.xlsx'. Bentuk tabel: (1633, 63)
